# Mistral LoRA Fine-Tuning

This notebook fine-tunes Mistral-7B-Instruct with LoRA for abstractive news summarization and evaluates the resulting summaries with ROUGE and BERTScore.

Expected setup:
- GPU runtime required
- Kaggle credentials available through environment variables
- `HUGGINGFACE_TOKEN` set before loading the model

In [ ]:
%%capture
%pip install -U transformers
#%pip install torch
%pip install -U peft
%pip install -i https://pypi.org/simple/ bitsandbytes
%pip install -U accelerate
%pip install -U trl
%pip install datasets==2.16.0
%pip install ipywidgets
    
%pip install peft
%pip install bert-score



In [ ]:
!pip install evaluate
import torch
import numpy as np
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import evaluate
!pip install rouge_score # Install the rouge_score package
import torch
from tqdm import tqdm
from bert_score import score 

In [ ]:
from warnings import filterwarnings
filterwarnings('ignore')
import json
import matplotlib.pyplot as plt
import re
import wandb
import os
import unicodedata
from pprint import pprint
import pandas as pd
import torch
from kaggle_secrets import UserSecretsClient
from datasets import Dataset, load_dataset
from huggingface_hub import notebook_login, login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model

import torch
import numpy as np
import evaluate
from tqdm.auto import tqdm
from transformers import get_scheduler
from torch.utils.data import DataLoader
from torch.optim import AdamW

import warnings

warnings.filterwarnings("ignore")


In [ ]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

In [ ]:
import os
# Set Kaggle credentials in the environment before running this notebook.
os.environ["KAGGLE_USERNAME"] = os.getenv("KAGGLE_USERNAME", "")
os.environ["KAGGLE_KEY"] = os.getenv("KAGGLE_KEY", "")


In [ ]:
 # Downloading dataset directly
!kaggle datasets download -d gowrishankarp/newspaper-text-summarization-cnn-dailymail --unzip -p /kaggle/working/

# Verify the downloaded files
!ls /kaggle/working/cnn_dailymail/


In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/cnn_dailymail/train.csv", nrows=3000)
df.columns = [str(q).strip() for q in df.columns]

dataset = Dataset.from_pandas(df)
dataset
df.head()

In [ ]:
!pip install datasets
from datasets import Dataset
df.columns = [str(q).strip() for q in df.columns]

dataset = Dataset.from_pandas(df)
dataset

In [ ]:
DEFAULT_SYSTEM_PROMPT = "Below is a news article. Write an insightful summary of the news article in 60 words or less. ".strip()

# generating prompts

def generate_training_prompt(
    article: str, summary: str, system_prompt: str = DEFAULT_SYSTEM_PROMPT
) -> str:
    bos_token = "<s>"
    eos_token = "</s>"

    full_prompt = ""
    full_prompt += bos_token
    full_prompt += "### Instruction:"
    full_prompt += "\n" + system_prompt
    full_prompt += "\n\n### Input:"
    full_prompt += "\n" + article
    full_prompt += "\n\n### Response:"
    full_prompt += "\n" + summary
    full_prompt += eos_token
    return full_prompt

In [ ]:
# creating prompts

def create_prompt(data_point):
    article, summary = data_point["article"], data_point["highlights"]
    article = unicodedata.normalize("NFKD", article)
    article = re.sub(r'\s(\.)\s', ' ', article)

    summary = re.sub(r'([^a-zA-Z\s])?\n(\w+)', r'\1 \2', summary)
    summary = re.sub(r'\s(\.)', '.', summary)
    return article.strip(), summary.strip()


def generate_text(data_point):
    article, summary = create_prompt(data_point)
    return generate_training_prompt(article, summary)


# Example usage with a new dataset format
example_data_point = {
    "id": "train_0",
    "article": df.iloc[5]["article"],
    "highlights": df.iloc[5]["highlights"]
}


example = generate_text(example_data_point)
print(example)

In [ ]:
# splitting into train-test set
dataset = dataset.shuffle(seed=42)
train_dataset = dataset.select(range(0, int(0.8 * len(dataset))))
test_dataset = dataset.select(range(int(0.8 * len(dataset)), len(dataset)))

In [ ]:
from huggingface_hub import notebook_login, login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
import torch
# Import LoRA-related classes
from peft import LoraConfig, get_peft_model

access_token = os.getenv("HUGGINGFACE_TOKEN")

def create_model_and_tokenizer():
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_enable_fp32_cpu_offload=True
    )

    # Loading the base model using your quantization configuration
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map='cuda:0',
        quantization_config=nf4_config,
        use_cache=False,
        token=access_token
    )
    model.config.use_cache = False

    # Defining the LoRA configuration.
    lora_config = LoraConfig(
        r=8,                        # Low-rank factor (adjust as needed)
        lora_alpha=32,              # Scaling factor (adjust as needed)
        lora_dropout=0.1,           # Dropout rate for LoRA layers
        target_modules=["q_proj", "v_proj"],  # Target modules to adapt; verify these for your model
        task_type="CAUSAL_LM"       # Task type for causal language modeling
    )
    # Wrapping the model with LoRA layers
    model = get_peft_model(model, lora_config)


    # Loading the tokenizer as before
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=access_token)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    return model, tokenizer


In [ ]:
def prepare_inputs_and_labels(article, summary):
    # Creating the full prompt by concatenating the instruction/article and the summary
    full_prompt = generate_training_prompt(article, summary)

    # Tokenize the full prompt
    tokenized_full = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    input_ids = tokenized_full.input_ids

    # Creating a prompt-only sequence to know where the summary starts
    prompt_only = generate_training_prompt(article, "")
    tokenized_prompt = tokenizer(
        prompt_only,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    # Counting the non-padding tokens to get the prompt length
    prompt_length = (tokenized_prompt.input_ids != tokenizer.pad_token_id).sum().item()


    labels = input_ids.clone()

    labels[:, :prompt_length] = -100
    return input_ids.to("cuda"), labels.to("cuda")


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Loading the ROUGE metric
rouge_metric = evaluate.load("rouge")

# Training function
def train_model(train_dataset, model, tokenizer, num_epochs=2, batch_size=2, learning_rate=5e-5):
    # Create DataLoader for training data
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Initializing optimizer and learning rate scheduler
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    num_training_steps = num_epochs * len(train_dataloader)
    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )
    
    # Set model to training mode
    model.train()
    
    # Training loop
    for epoch in range(num_epochs):
        epoch_loss = 0
        
        for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}/{num_epochs}"):
            inputs_list = []
            labels_list = []
            for article, summary in zip(batch['article'], batch['highlights']):
                input_ids, labels = prepare_inputs_and_labels(article, summary, tokenizer)  
                inputs_list.append(input_ids)
                labels_list.append(labels)

            inputs = torch.cat(inputs_list, dim=0)
            labels = torch.cat(labels_list, dim=0)

            # Forward pass
            outputs = model(input_ids=inputs, labels=labels)

            # Assertion to catch bad labels
            vocab_size = model.config.vocab_size
            assert torch.all((labels == -100) | ((labels >= 0) & (labels < vocab_size))), "Labels contain invalid token IDs"

            loss = outputs.loss
            epoch_loss += loss.item()

            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {epoch_loss/len(train_dataloader):.4f}")
        torch.cuda.empty_cache()
    
    
    return model

In [ ]:
# Evaluation function with BERTScore
def evaluate_model(test_dataset, model, tokenizer, batch_size=2):
    model.eval()
    
    generated_summaries = []
    gold_summaries = []
    
    # Processing the test dataset in batches to generate summaries
    for i in tqdm(range(0, len(test_dataset), batch_size), desc="Evaluating"):
        # Slicing the dataset and converting it into a dict
        batch = test_dataset[i : i + batch_size]
        batch_summaries = generate_batch_summaries(batch, model, tokenizer)
        generated_summaries.extend(batch_summaries)
        gold_summaries.extend([example.strip() for example in batch['highlights']])
    
    # Calculating ROUGE scores
    results = rouge_metric.compute(predictions=generated_summaries, references=gold_summaries)
    
    # Calculating BERTScore
    P, R, F1 = score(generated_summaries, gold_summaries, lang="en", verbose=True, device="cuda")
    bert_score_avg = F1.mean().item() if isinstance(F1, torch.Tensor) else F1
    results["bert_score_f1"] = bert_score_avg

    print("Evaluation Results:")
    print(results)
    print(f"BERT Score F1: {bert_score_avg:.4f}")

    torch.cuda.empty_cache()
    return results


In [ ]:
# Modifying generate_batch_summaries to accept model and tokenizer as parameters
def generate_batch_summaries(batch, model, tokenizer):
    # Creating prompts for each example in the batch
    prompts = [generate_training_prompt(batch['article'][i], "") for i in range(len(batch['article']))]
    # Removing the end-of-sequence token if present
    prompts = [p[:-len("</s>")] if p.endswith("</s>") else p for p in prompts]

    # Tokenizing the batch with padding
    inputs = tokenizer(prompts, return_tensors="pt", truncation=True, padding=True, max_length=512).to("cuda")

    # Generating summaries without computing gradients
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=50,        # Adjust based on desired summary length
            do_sample=True,           # Set to False for deterministic output
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decoding the generated tokens
    outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    # Removing the prompt text from the output
    summaries = [output.replace(prompt, "").strip() for prompt, output in zip(prompts, outputs)]
    torch.cuda.empty_cache()
    return summaries

In [ ]:
# Training the model for a given number of epochs while tracking average training loss per epoch.
def train_for_epochs(train_dataset, model, tokenizer, num_epochs, batch_size, learning_rate):

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_losses = []
    
    # Loop over epochs
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        batch_count = 0
        # Processing training dataset in batches
        for i in tqdm(range(0, len(train_dataset['article']), batch_size), desc=f"Training Epoch {epoch+1}"):
            # Create batch dictionary
            batch = test_dataset[i : i + batch_size]
            if len(batch['article']) == 0:
                continue  # Skip empty batch
            optimizer.zero_grad()
            batch_loss = 0.0
            
            # Processing each example in the batch
            for article, summary in zip(batch['article'], batch['highlights']):
                # Preparing inputs and labels for the example
                input_ids, labels = prepare_inputs_and_labels(article, summary, tokenizer)
                outputs = model(input_ids=input_ids, labels=labels)
                loss = outputs.loss
                loss.backward()
                batch_loss += loss.item()
            optimizer.step()
            
            # Average loss for the batch (averaged across examples)
            avg_batch_loss = batch_loss / len(batch['article'])
            epoch_loss += avg_batch_loss
            batch_count += 1

        avg_epoch_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
        train_losses.append(avg_epoch_loss)
        print(f"Epoch {epoch+1}/{num_epochs}: Training Loss = {avg_epoch_loss:.4f}")
    
    return model, train_losses


#Evaluating the model on the given dataset by computing the average loss.
    
    
def evaluate_loss(dataset, model, tokenizer, batch_size=2):

    model.eval()
    total_loss = 0.0
    batch_count = 0
    # Process the dataset in batches
    for i in tqdm(range(0, len(dataset['article']), batch_size), desc="Evaluating Loss"):
        batch = test_dataset[i : i + batch_size]
        if len(batch['article']) == 0:
            continue  # Skip empty batch
        batch_loss = 0.0
        # Process each example in the batch
        for article, summary in zip(batch['article'], batch['highlights']):
            input_ids, labels = prepare_inputs_and_labels(article, summary, tokenizer)
            with torch.no_grad():
                outputs = model(input_ids=input_ids, labels=labels)
                loss = outputs.loss
                batch_loss += loss.item()
        avg_batch_loss = batch_loss / len(batch['article'])
        total_loss += avg_batch_loss
        batch_count += 1

    average_loss = total_loss / batch_count if batch_count > 0 else None
    return average_loss

In [ ]:
# Modifying prepare_inputs_and_labels to accept tokenizer as an argument
def prepare_inputs_and_labels(article, summary, tokenizer):
    # Generate full prompt (article + summary)
    full_prompt = generate_training_prompt(article, summary)
    
    # Tokenizing with truncation and padding
    tokenized = tokenizer(
        full_prompt,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    input_ids = tokenized["input_ids"]

    # Generating the prompt-only part to mask it in the loss
    prompt_only = generate_training_prompt(article, "")
    tokenized_prompt = tokenizer(
        prompt_only,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    prompt_length = (tokenized_prompt["input_ids"] != tokenizer.pad_token_id).sum().item()

    # Set labels: ignore the prompt tokens by assigning -100
    labels = input_ids.clone()
    labels[:, :prompt_length] = -100  # ignore prompt tokens in loss

    # Send to CUDA
    return input_ids.to("cuda"), labels.to("cuda")

In [ ]:
# Main execution
def main():
    model, tokenizer = create_model_and_tokenizer() 
    
    # Set parameters
    num_epochs = 2
    batch_size = 2
    learning_rate = 5e-5
    
    # Train the model
    print("Starting training phase...")

    model, train_losses = train_for_epochs(train_dataset, model, tokenizer, num_epochs, batch_size, learning_rate)

    # Evaluating loss on test dataset
    test_loss = evaluate_loss(test_dataset, model, tokenizer, batch_size)
    print(f"Average Test Loss: {test_loss:.4f}")

    
    # Evaluating the model
    print("\nStarting evaluation phase...")
    evaluation_results = evaluate_model(test_dataset, model, tokenizer, batch_size)

    
    torch.cuda.empty_cache()

    # Saving the trained model
    model.save_pretrained("./trained_summarization_model")
    tokenizer.save_pretrained("./trained_summarization_model")
    print("Model and tokenizer saved to ./trained_summarization_model")

In [ ]:
if __name__ == "__main__":
    main()


In [ ]:
def print_trainable_parameters(model):
    # Counting all parameters and trainable ones (parameters with requires_grad=True)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print("Total parameters in model:", total_params)
    print("Trainable parameters (LoRA parameters):", trainable_params)
    print("Percentage of trainable parameters: {:.2f}%".format(100 * trainable_params / total_params))


model, tokenizer = create_model_and_tokenizer()  # Your function that loads & wraps the model with LoRA

# Printing the number of trainable parameters (which will primarily be your LoRA layers)
print_trainable_parameters(model)
